# V1 — Konya SAR Soil Characterization: Exploratory Report

**AOI:** Konya agricultural plain (~23×20 km)  
**Wet season:** 15 Jan – 15 Mar 2024  
**Dry season:** 15 Jul – 15 Sep 2024  
**SAR source:** Sentinel-1 IW σ⁰ VV+VH via Sentinel Hub CDSE  
**DEM source:** Copernicus GLO-30 via Sentinel Hub CDSE  

In [ ]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import rasterio

INTERIM   = Path('../data/interim')
PROCESSED = Path('../data/processed')

def read(path, band=1, nodata=-9999.0):
    with rasterio.open(path) as src:
        d = src.read(band).astype(np.float32)
        nd = src.nodata if src.nodata is not None else nodata
    d[d == nd] = np.nan
    return d

vv_wet   = read(INTERIM / 's1/s1_vv_konya_wet.tif')
vv_dry   = read(INTERIM / 's1/s1_vv_konya_dry.tif')
vh_wet   = read(INTERIM / 's1/s1_vh_konya_wet.tif')
nddi     = read(INTERIM / 'indices/nddi.tif')
ratio_w  = read(INTERIM / 'indices/vv_vh_ratio_wet.tif')
twi      = read(INTERIM / 'hydrology/twi.tif')
slope    = read(INTERIM / 'terrain/slope.tif')
classes  = read(PROCESSED / 'surface_response_classes_konya.tif').astype(float)
classes[classes == 255] = np.nan
risk     = read(PROCESSED / 'construction_risk_konya.tif')
stats    = json.loads((PROCESSED / 'stats_v1.json').read_text())

print('Data loaded.')

## 1. SAR Backscatter Maps

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Sentinel-1 VV Backscatter (σ⁰ dB)', fontsize=13, fontweight='bold')

for ax, data, title in [
    (axes[0], vv_wet, 'Wet Season (Jan–Mar 2024)'),
    (axes[1], vv_dry, 'Dry Season (Jul–Sep 2024)'),
    (axes[2], vv_wet - vv_dry, 'Difference (Wet − Dry)'),
]:
    vmin, vmax = (-25, -5) if 'Diff' not in title else (-5, 5)
    cmap = 'gray' if 'Diff' not in title else 'RdBu'
    im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax)
    plt.colorbar(im, ax=ax, label='dB', shrink=0.8)
    ax.set_title(title); ax.axis('off')

plt.tight_layout(); plt.show()

print(f'VV wet:  mean={np.nanmean(vv_wet):.2f} dB  std={np.nanstd(vv_wet):.2f}')
print(f'VV dry:  mean={np.nanmean(vv_dry):.2f} dB  std={np.nanstd(vv_dry):.2f}')
print(f'Δ VV:    mean={np.nanmean(vv_wet-vv_dry):.2f} dB')

## 2. NDDI — Normalized Difference Dryness Index

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im = axes[0].imshow(nddi, cmap='RdBu', vmin=-0.4, vmax=0.4)
plt.colorbar(im, ax=axes[0], label='NDDI', shrink=0.8)
axes[0].set_title('NDDI = (σ_wet − σ_dry) / (σ_wet + σ_dry)\nBlue=wet dominant  Red=dry dominant')
axes[0].axis('off')

v = nddi[~np.isnan(nddi)].flatten()
axes[1].hist(v, bins=80, color='steelblue', edgecolor='white', linewidth=0.3)
axes[1].axvline(0, color='red', lw=1.5, linestyle='--', label='neutral')
axes[1].axvline(np.nanmean(v), color='orange', lw=1.5, label=f'mean={np.nanmean(v):.3f}')
axes[1].set_xlabel('NDDI'); axes[1].set_ylabel('Pixel count')
axes[1].set_title('NDDI Distribution')
axes[1].legend()

plt.tight_layout(); plt.show()

n = stats['nddi_summary']
print(f'Wet dominant  (NDDI > +0.05): {n["pct_positive"]:.1f}%')
print(f'Neutral       (|NDDI| ≤ 0.05): {n["pct_neutral"]:.1f}%')
print(f'Dry dominant  (NDDI < -0.05): {n["pct_negative"]:.1f}%')

## 3. Terrain Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im = axes[0].imshow(twi, cmap='Blues',
    vmin=np.nanpercentile(twi, 5), vmax=np.nanpercentile(twi, 95))
plt.colorbar(im, ax=axes[0], label='TWI', shrink=0.8)
axes[0].set_title(f'Topographic Wetness Index\nmean={np.nanmean(twi):.2f}')
axes[0].axis('off')

im = axes[1].imshow(slope, cmap='hot_r', vmin=0, vmax=5)
plt.colorbar(im, ax=axes[1], label='°', shrink=0.8)
axes[1].set_title(f'Slope (degrees)\nmean={np.nanmean(slope):.2f}°  p90={np.nanpercentile(slope,90):.2f}°')
axes[1].axis('off')

plt.tight_layout(); plt.show()

## 4. Statistical Analysis

In [ ]:
print('=== SPEARMAN RANK CORRELATIONS ===')
print(f'{"Variable pair":<30} {"ρ":>8}  {"p-value":>12}  Interpretation')
print('-'*75)
labels = {
    'nddi_vs_twi':   'NDDI vs TWI',
    'nddi_vs_slope': 'NDDI vs Slope',
    'vv_wet_vs_dry': 'VV wet vs VV dry',
    'vv_wet_vs_twi': 'VV wet vs TWI',
}
for k, lbl in labels.items():
    r = stats['spearman'][k]
    rho, p = r['rho'], r['p_value']
    interp = 'weak' if abs(rho)<0.1 else ('moderate' if abs(rho)<0.4 else 'strong')
    print(f'{lbl:<30} {rho:>+8.4f}  {p:>12.2e}  {interp}')

print()
print('=== KRUSKAL-WALLIS TESTS ===')
print(f'{"Test":<40} {"H":>10}  Significant?')
print('-'*60)
kw_labels = {
    'vv_wet_by_twi_quantile': 'VV wet by TWI quartile',
    'vv_dry_by_twi_quantile': 'VV dry by TWI quartile',
    'nddi_by_twi_quantile':   'NDDI by TWI quartile',
    'vv_wet_by_slope':        'VV wet by slope quartile',
}
for k, lbl in kw_labels.items():
    r = stats['kruskal_wallis'][k]
    sig = '✓ YES (p<0.001)' if r['p_value'] < 0.001 else '✗ NO'
    print(f'{lbl:<40} {r["statistic"]:>10.1f}  {sig}')

## 5. Surface Response Classification

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cmap5 = mcolors.ListedColormap(['#d73027','#fc8d59','#fee090','#91bfdb','#4575b4'])
norm5 = mcolors.BoundaryNorm([-0.5,0.5,1.5,2.5,3.5,4.5], cmap5.N)
im = axes[0].imshow(classes, cmap=cmap5, norm=norm5)
cbar = plt.colorbar(im, ax=axes[0], ticks=[0,1,2,3,4], shrink=0.8)
cbar.ax.set_yticklabels(['Dry','Mod.Dry','Trans.','Seas.Wet','Pers.Wet'], fontsize=8)
axes[0].set_title('Surface Response Classes\n(k-means on NDDI + TWI + Slope)')
axes[0].axis('off')

cls_names = ['Dry','Mod.Dry','Trans.','Seas.Wet','Pers.Wet']
colors5   = ['#d73027','#fc8d59','#fee090','#91bfdb','#4575b4']
counts    = [(classes == i).sum() for i in range(5)]
total     = sum(counts)
axes[1].barh(cls_names, [100*c/total for c in counts], color=colors5)
axes[1].set_xlabel('%'); axes[1].set_title('Class Distribution')
for i, c in enumerate(counts):
    axes[1].text(100*c/total+0.3, i, f'{100*c/total:.1f}%', va='center')

plt.tight_layout(); plt.show()

## 6. Construction Moisture Risk

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im = axes[0].imshow(risk, cmap='RdYlGn_r', vmin=0, vmax=0.8)
plt.colorbar(im, ax=axes[0], label='Risk score [0–1]', shrink=0.8)
axes[0].set_title(f'Construction Moisture Risk Index\nmean={np.nanmean(risk):.2f}  max={np.nanmax(risk):.2f}')
axes[0].axis('off')

v = risk[~np.isnan(risk)].flatten()
axes[1].hist(v, bins=60, color='tomato', edgecolor='white', linewidth=0.3)
axes[1].set_xlabel('Risk score'); axes[1].set_ylabel('Pixel count')
axes[1].set_title('Risk Distribution')
for threshold, color, label in [(0.3,'orange','Moderate >0.3'),(0.5,'red','High >0.5')]:
    pct = 100*(v>threshold).mean()
    axes[1].axvline(threshold, color=color, lw=2, linestyle='--', label=f'{label} ({pct:.1f}%)')
axes[1].legend()

plt.tight_layout(); plt.show()

## 7. Key Findings

### SAR Signal
- Mean VV wet = **-12.97 dB**, dry = **-12.79 dB** — only **0.18 dB** seasonal difference  
- Small difference likely due to winter wheat masking soil moisture signal in wet season  
- Dry season has higher variance (std 3.13 vs 2.32) — more diverse surface conditions after harvest

### NDDI
- **44.6%** dry-dominant pixels (irrigated summer crops / volume scattering in dry season)  
- **41.3%** wet-dominant pixels (bare soil higher moisture in winter)  
- Near-zero mean (−0.018) — balanced seasonal contrast in this AOI

### Terrain
- Extremely flat: mean slope = **1.05°**, 90th percentile = **2.14°**  
- TWI range valid for flat agriculture, though terrain-driven drainage signal is weak

### Statistics
- All Kruskal-Wallis tests **highly significant** (H > 4000, p < 0.001)  
  → backscatter distributions differ meaningfully across TWI and slope strata  
- VV wet vs VV dry Spearman ρ = **+0.61** — good inter-season consistency per pixel  
- NDDI vs TWI ρ = **−0.025** — very weak terrain-moisture correlation (expected in flat plains)

### Limitations
- No VH-based polarimetric analysis (VH available but analysis pending)  
- SoilGrids reference labels not yet incorporated (V2 target)  
- Seasonal contrast could be improved: Nisan–Mayıs as wet reference may be better
